# Titanic Data Workflow

**Author:** Francisco Mateus

**Dataset:** Titanic: Machine Learning from Disaster (Kaggle)

This notebook builds a reproducible workflow over the 891-row Titanic passenger table, it does so by meausring what is missing, cleaning the table with reusable functions, exploring survival patterns, drawing labelled figures, closing with a summary that will goes through the results.

## How this notebook is organised

Six sections, in the order the project asks for them:

1. **Setup**: the libraries and the versions this run used.
2. **Ingestion**: the CSV loaded from disk, with its provenance.
3. **Assessment and Cleaning**: what is missing, and the functions that repair it.
4. **Exploratory Analysis**: survival rates by sex, class, title and deck.
5. **Visualizations**: the labelled figures, each read in the cell below it.
6. **Summary**: what the numbers support and what they do not.

Every number quoted in the written report is printed by a cell in this notebook.


## 1. Setup

In [1]:
import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")

print(f"numpy {np.__version__} | pandas {pd.__version__} | "
      f"matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")

numpy 2.5.3 | pandas 3.0.6 | matplotlib 3.11.2 | seaborn 0.13.2


## 2. Ingestion

The dataset sits beside this notebook as `titanic.csv`, 
so the load is local, making it possible to run without internet connection.
The comment records where the file came from.


In [2]:
# Provenance: Kaggle "Titanic - Machine Learning from Disaster",
# mirror https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
df = pd.read_csv("titanic.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### What the rows mean

One row is one passenger, and each column is one fact about that passenger.

| Column | Kind | Meaning |
|---|---|---|
| `PassengerId` | number | Row identifier; it says nothing about the person |
| `Survived` | `0` or `1` | The outcome to explain: `1` survived, `0` did not |
| `Pclass` | `1`, `2`, `3` | Ticket class: `1` first, `2` second, `3` third, with no fourth |
| `Name` | text | Full name; carries the honorific that becomes `title` during cleaning |
| `Sex` | text | `male` or `female` |
| `Age` | number | Age in years; blanks are counted in the Assessment section |
| `SibSp` | number | Siblings and spouses aboard |
| `Parch` | number | Parents and children aboard |
| `Ticket` | text | Ticket number |
| `Fare` | number | Price paid for the ticket |
| `Cabin` | text | Cabin code; blank for many rows |
| `Embarked` | text | Port of boarding: `S` Southampton, `C` Cherbourg, `Q` Queenstown |


### Check the load

Before analysing anything, confirm the table arrived intact:

- `891` rows and `12` columns, the size the project expects;
- numeric columns stored as numbers, `int64` or `float64`;
- text columns stored as text, `str`;
- an assertion on the shape, so a different file fails the run instead of
  producing plausible numbers.

The next cell prints each check and then runs the assertion.


In [3]:
print(f"rows x columns: {df.shape}")
print(f"columns: {list(df.columns)}")
df.dtypes.to_frame("dtype")

assert df.shape == (891, 12), f"unexpected shape: {df.shape}"
print("load check: 891 rows, 12 columns")


rows x columns: (891, 12)
columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
load check: 891 rows, 12 columns
